### 1. Chain

In [2]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
load_dotenv()

prompt = PromptTemplate(
    template='Generate 5 interesting facts about {topic}',
    input_variables=['topic']
)

model = ChatOpenAI()
parser = StrOutputParser()
chain = prompt | model | parser
result = chain.invoke({'topic':'cricket'})
print(result)

chain.get_graph().print_ascii()

c:\Users\sidha\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


1. Cricket is one of the oldest team sports in the world, with origins dating back to the 16th century in south-east England. It is believed to have been played by children in the Weald region of Kent as early as 1550.

2. The longest test match in cricket history was played between England and South Africa in 1939. The match lasted for a total of 9 days, with all eleven days of the game featuring actual play. The match was eventually declared a draw due to time constraints.

3. The highest individual score ever recorded in a test match was achieved by Brian Lara of the West Indies in 1994. Lara scored an incredible 400 not out against England, surpassing the previous record of 380 set by Matthew Hayden.

4. Cricket is known as the gentleman's game due to its emphasis on fair play, respect for opponents, and adherence to the spirit of the game. Players are expected to uphold the values of sportsmanship on and off the field.

5. One of the most famous cricket grounds in the world is Lor

### 2. Parallal Chain

In [5]:
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

load_dotenv()

model1 = ChatOpenAI()
model2 = ChatOpenAI()

prompt1 = PromptTemplate(
    template='Generate short and simple notes from the following text \n {text}',
    input_variables=['text']
)

prompt2 = PromptTemplate(
    template='Generate 5 short question answers from the following text \n {text}',
    input_variables=['text']
)

prompt3 = PromptTemplate(
    template='Merge the provided notes and quiz into a single document \n notes -> {notes} and quiz -> {quiz}',
    input_variables=['notes', 'quiz']
)

parser = StrOutputParser()

parallel_chain = RunnableParallel({
    'notes': prompt1 | model1 | parser,
    'quiz': prompt2 | model2 | parser
})

merge_chain = prompt3 | model1 | parser
chain = parallel_chain | merge_chain

text = """
Support vector machines (SVMs) are a set of supervised learning methods used for classification, regression and outliers detection.
The advantages of support vector machines are:
Effective in high dimensional spaces.
Still effective in cases where number of dimensions is greater than the number of samples.
Uses a subset of training points in the decision function (called support vectors), so it is also memory efficient.
Versatile: different Kernel functions can be specified for the decision function. Common kernels are provided, but it is also possible to specify custom kernels.
The disadvantages of support vector machines include:
If the number of features is much greater than the number of samples, avoid over-fitting in choosing Kernel functions and regularization term is crucial.
SVMs do not directly provide probability estimates, these are calculated using an expensive five-fold cross-validation (see Scores and probabilities, below).
The support vector machines in scikit-learn support both dense (numpy.ndarray and convertible to that by numpy.asarray) and sparse (any scipy.sparse) sample vectors as input. However, to use an SVM to make predictions for sparse data, it must have been fit on such data. For optimal performance, use C-ordered numpy.ndarray (dense) or scipy.sparse.csr_matrix (sparse) with dtype=float64.
"""

result = chain.invoke({'text': text})
print(result)
chain.get_graph().print_ascii()

Support Vector Machines (SVMs) are used for classification, regression, and outliers detection in supervised learning. They are effective in high dimensional spaces and still effective when the number of dimensions is greater than the number of samples. Some advantages of SVMs include memory efficiency, versatility with different kernel functions, and the ability to handle both dense and sparse sample vectors as input.

However, a disadvantage of SVMs when the number of features is much greater than the number of samples is the risk of overfitting, which requires careful selection of kernel functions and regularization terms. Additionally, SVMs do not provide direct probability estimates, requiring users to calculate them using an expensive five-fold cross-validation method.

In scikit-learn, support vector machines support both dense and sparse sample vectors as input. For optimal performance when working with sparse data, it is recommended to use C-ordered numpy.ndarray or scipy.spar

### 2. Conditional Chain

In [7]:
from langchain_core.runnables import RunnableBranch, RunnableLambda
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import Literal

class Feedback(BaseModel):
    sentiment: Literal["positive", "negative"] = Field(
        description="Give the sentiment of the feedback"
    )

parser2 = PydanticOutputParser(pydantic_object=Feedback)

prompt1 = PromptTemplate(
    template='Classify the sentiment of the following feedback text into positive or negative \n {feedback} \n {format_instruction}',
    input_variables=['feedback'],
    partial_variables={'format_instruction': parser2.get_format_instructions()}
)

classifier_chain = prompt1 | model | parser2

prompt2 = PromptTemplate(
    template='Write an appropriate response to this positive feedback \n {feedback}',
    input_variables=['feedback']
)

prompt3 = PromptTemplate(
    template='Write an appropriate response to this negative feedback \n {feedback}',
    input_variables=['feedback']
)

branch_chain = RunnableBranch(
    (lambda x: x.sentiment == 'positive', prompt2 | model | parser),
    (lambda x: x.sentiment == 'negative', prompt3 | model | parser),
    RunnableLambda(lambda x: "could not find sentiment")
)

chain = classifier_chain | branch_chain

print(chain.invoke({'feedback': 'This is a beautiful phone'}))
chain.get_graph().print_ascii()

Thank you so much for your kind words! I'm thrilled to hear that you had a positive experience. Let me know if there's anything else I can help with.
    +-------------+      
    | PromptInput |      
    +-------------+      
            *            
            *            
            *            
   +----------------+    
   | PromptTemplate |    
   +----------------+    
            *            
            *            
            *            
     +------------+      
     | ChatOpenAI |      
     +------------+      
            *            
            *            
            *            
+----------------------+ 
| PydanticOutputParser | 
+----------------------+ 
            *            
            *            
            *            
       +--------+        
       | Branch |        
       +--------+        
            *            
            *            
            *            
    +--------------+     
    | BranchOutput |     
    +-------------

### 4. Sequential Chain

In [ ]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

prompt1 = PromptTemplate(
    template='Generate a detailed report on {topic}',
    input_variables=['topic']
)

prompt2 = PromptTemplate(
    template='Generate a 5 pointer summary from the following text \n {text}',
    input_variables=['text']
)

model = ChatOpenAI()
parser = StrOutputParser()
chain = prompt1 | model | parser | prompt2 | model | parser
result = chain.invoke({'topic': 'Unemployment in India'})

print(result)
chain.get_graph().print_ascii()

Summary:
1. The current unemployment rate in India is around 7.1%, with urban areas experiencing higher rates than rural areas.
2. Causes of unemployment include rapid population growth, lack of skills, technological disruption, the prevalence of the informal economy, and economic slowdowns.
3. Government initiatives to address unemployment include skill development programs, initiatives to boost domestic manufacturing and entrepreneurship, and the MGNREGA scheme.
4. Recommendations for tackling unemployment include encouraging entrepreneurship, investing in education, supporting MSMEs, and promoting public-private partnerships.
5. Tackling unemployment in India requires a comprehensive approach involving government policies, private sector collaboration, and individual efforts to unlock the country's human capital potential.
     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *         